# Data Exploration and Cleaning

## DuckDB
- **DuckDB** is an analytical database designed for fast data analysis
- You can think of it as SQLite, but optimized for analytical workloads (OLAP).

## Why databases beat spreadsheets
### Why would a spreadsheet be problematic?
Imagine putting everything into one Excel spreadsheet.

You might have something like this:
| Building ID | District | Material | Damage | Occupants |
| --- | --- | --- | --- | --- |
| 101 | Kathmandu | Brick | Severe | 5 |
| 102 | Lalitpur | Concrete | Moderate | 3 |
| 103 | Kathmandu | Brick | Minor | 4 |
| 104 | Bhaktapur | Wood | Severe | 6 |


- You can separate data into different sheets.
- You can use IDs to connect information manually.
- However, maintaining relationships and enforcing data integrity constraints is more difficult.

1. Large datasets : Excel has a maximum of 1,048,576 rows per worksheet.
2. Data duplication: Data duplication

    Suppose 10,000 buildings are located in Kathmandu.

        You might repeat the district name 10,000 times.

            If the district name changes or is entered inconsistently, you could end up with:
            - Kathmandu
            - kathmandu
            - Katmandu
            These could be interpreted as three different districts during analysis.

3. Data inconsistency

    Imagine the same building appears in two different spreadsheets.

    One spreadsheet says its building material is Brick, while another says Concrete.

    Which one is correct?

    Updating one spreadsheet doesn't automatically update the other.

    This creates inconsistent information.

### How databases solve these problems: Normalization
Normalization is the process of organizing data into related tables to reduce unnecessary duplication and maintain data consistency.

Instead of storing everything in one large table, we separate the information.

For example:

### Table 1: Buildings

| building_id | material | floors |
| --- | --- | --- |
| 101 | Brick | 3 |
| 102 | Concrete | 5 |
| 103 | Brick | 2 |

---

### Table 2: Districts

| district_id | district_name |
| --- | --- |
| 1 | Kathmandu |
| 2 | Lalitpur |
| 3 | Bhaktapur |

---

### Table 3: Building locations

| building_id | district_id |
| --- | --- |
| 101 | 1 |
| 102 | 2 |
| 103 | 1 |

---

### Table 4: Damage assessments

| building_id | damage_level |
| --- | --- |
| 101 | Severe |
| 102 | Moderate |
| 103 | Minor |

Notice that we don't repeatedly store the district name in the building location table. We store its ID instead.

#### Benefits of normalization:

- No redundancy: Redundancy means storing the same information repeatedly when it doesn't need to be repeated. 
  
    | Order ID | Customer name | Address | Product |
    | --- | --- | --- | --- |
    | 101 | John | Nairobi | Laptop |
    | 102 | John | Nairobi | Phone |
    | 103 | Mary | Mombasa | Tablet |
    | 104 | John | Nairobi | Mouse |

    Notice that John's address is repeated three times.

    If John changes his address, you have to update three rows.

    With normalization, you separate customers and orders.

- Consistency: Update one row, every query instantly reflects the change across the entire database
  
   Consistency means that the same information does not have conflicting values in different places.

    Imagine John moves from Nairobi to Kisumu.

    In the unnormalized table, you might update only two of his three order records.
- Integrity: 
-   Integrity means ensuring that the data follows rules and relationships remain valid.
    A database can enforce a foreign key constraint to ensure that:
    You cannot create an order for a customer who doesn't exist.
    You cannot delete a customer who still has orders,
    Constraints prevent orphaned orders() : orphaned record is like a order of customer id 5 but customer id 5 does not exist
- Clarity: "How many customers named John?" is unambiguous — one row per customer

##### How databases ensure data integrity(Data Integrity Constrains)
1. PRIMARY KEY: Uniquely identifies each record (row) in a table. It cannot contain NULL or duplicate values.
2. FOREIGN KEY: References a primary key (or another eligible unique key) in another table to maintain relationships between tables.
3. NOT NULL: Ensures that a column cannot contain NULL values.
4. CHECK: Ensures that values meet a specified condition.
   - Example: age >= 0
5. UNIQUE: Ensures that values in a column (or combination of columns) are not duplicated.
6. DEFAULT: Automatically assigns a predefined value when no value is provided for a column.


## Basic database terminology
1. **Schema**:A blueprint or structure of a database. It defines which tables exist, what data each table stores, the columns, data types and relationships between tables.
2. **Table**: Organizes data into rows and columns.
3. **Query**: An SQL command used to retrieve, insert, update or delete data in a database.
4. **Index**: A data structure that helps a database locate records faster without scanning the entire table.
5. **Transaction**: Group of operations that succeeds or fail together

The bridge table pattern: when two tables have a many-to-many relationship, databases use a third table — a bridge or junction table — that holds both foreign keys

### Entity-Relationship (ER) Diagrams

**ERD (Entity Relationship Diagram)**: A visual representation of a database that shows:
1. Entities: The tables in the database.
2. Attributes: The columns or fields in each table.
3. Relationships: How the tables are connected.
4. Keys: Primary keys (PK) and foreign keys (FK) used to identify records and establish relationships.

## SQL Execution Order
The mental model: think of SQL execution as a pipeline — you open the table, filter it, group it, filter groups, then decide what to show. The output step (SELECT) always comes after all the filtering steps.

## Why DuckDB Wins at Analytics
1. Columnar storage
   This is one of DuckDB's biggest advantages.
   - Row-based storage: Stores data row by row.
   - Columnar storage: Stores data column by column.
   - Imagine a table with 1 million hospital records and 10 columns. You want to calculate the average length of hospital stay.
    ```
    SELECT AVG(length_of_stay)
    FROM hospital_records;
    ```

    A row-oriented database may need to read through rows containing all 10 columns.

    DuckDB's columnar storage allows it to focus on the length_of_stay column, avoiding unnecessary column reads.
    ![duckdb_sql](../images/Screenshot%202026-11-03%20at%2008.45.10.png)
2. Fast analytical queries
    DuckDB uses vectorized query execution.

    Instead of processing one row at a time, it processes batches of values together.
3. Works directly with data files
    DuckDB can query CSV, Parquet and other supported file formats without first importing everything into a traditional database.
    No server required
4. DuckDB is an embedded database.

    This means you don't need to set up and manage a separate database server before using it.
5. Integrates with Python
    DuckDB works well with Python libraries such as Pandas and Polars.

    You can query data using SQL and continue analysing it using Python.